# Component 2 - Line Win %

Gives a win % for every player prop line the books posted, using `scripts/line_model.py`. It needs the feature file (*Feature Engineering*) and the historical odds (`fetch_historical_odds.py`). Takes about 2 minutes.

**The model uses stats only.** It's the *Stat Predictions* model: last season, trained on the first half, tested on the second half. Each predicted stat becomes a distribution, and the win % for a side is the chance the stat lands on that side of the line. The spread of that distribution is fitted on the last two weeks of the train half, so nothing from the test half is used. Odds are only used afterwards, to compare.

**Each row is one side (Over or Under) of one line:**
- **avg_vegas_odds / vegas_implied_p**: the books' average American odds for that side, and the win rate those odds need (vig included, so the two sides add up to more than 100%).
- **vegas_p_win**: the same with the vig removed (each book's two sides scaled to add up to 100%, then averaged): the books' estimate of the real chance.
- **model_mean / model_p_win / model_odds**: the predicted stat, the model's chance this side wins, and the American odds that chance is worth (its fair price).
- **best_odds / best_book / breakeven_p**: the best price across books, where, and the win rate it needs.
- **avg_vegas_odds_Decimal / best_odds_Decimal**: the same average and best odds in decimal (total return per $1, stake included).
- **edge**: model minus Vegas win chance. **ev**: model's expected profit per $1 at the best odds.
- **actual / outcome / profit_per_1**: the stat, Win/Loss/Push, and the profit on $1 at the best odds.
- **main_line**: the line most books quoted for that player and stat (the others are alternate lines).

In [ ]:
LEAGUE = "nba"

In [ ]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "line_model.py").exists())
sys.path.insert(0, str(root / "scripts"))

import importlib
import fetch_player_stats, build_features, stat_model, line_model as lm
for m in (fetch_player_stats, build_features, stat_model, lm):
    importlib.reload(m)  # pick up script edits without restarting the kernel

import pandas as pd
pd.set_option("display.max_columns", None)

## Train the stats model and price every line

In [ ]:
out = lm.run(LEAGUE)
bets = out["bets"]

## Bet by bet

In [ ]:
cols = ["game_date", "player", "team", "opponent", "market", "line", "side", "books",
        "avg_vegas_odds", "vegas_implied_p", "vegas_p_win", "model_mean", "model_p_win", "model_odds",
        "best_odds", "best_book", "edge", "ev", "actual", "outcome"]
bets.loc[bets["main_line"], cols].round(3).head(30)

In [ ]:
PLAYER = "Josh Giddey"
bets.loc[(bets["player"] == PLAYER) & bets["main_line"], cols].round(3)

## Is the model's win % accurate?

Brier score on main lines, model vs the books' average (lower is better):

In [ ]:
lm.accuracy(bets).round(4)

When the model says X% to win, how often did the side actually win? Compare with the books:

In [ ]:
print("Model")
display(lm.calibration(bets).round(3))
print("Vegas")
display(lm.calibration(bets, "vegas_p_win").round(3))

## Would betting the model's edges have paid?

$1 flat on every main-line bet in each ev range, at the best odds:

In [ ]:
lm.ev_buckets(bets).round(3)

## Save

In [ ]:
lm.save(LEAGUE, out)